# Lab 5: Arithmetic and Number Representations
## Machine Learning Hardware Course

This notebook is Part 1 of the Arithmetic and Number Representations lab. It covers:
1. Environment setup
2. Quantize from FP32 to int8/6/4, FP16, BF16, FP8_E4M3, FP8_E5M2. Then present the result in bin, and reconstruct back to FP32.

## 📋 Lab To-Do Summary — 10 Items

Replace every `None` with the correct one-liner. Use the `# Hint:` directly above as your guide.
Record experiment results in your **printed worksheet** as you run each section.

| # | Cell | Location | Hint | Worksheet |
|---|------|----------|------|-----------|
| 1 | INT cell | for bits loop `Call quantize() on fp32_values | q_vals = quantize(fp32_values, bits) | Part 2 Q1 table |
| 2 | INT cell | Error loop — MAE | `np.mean(np.abs(fp32_values - dq_vals))` | Part 2 Q1 table |
| 3 | INT cell | Error loop — MSE | `np.mean((fp32_values - dq_vals) ** 2)` | Part 2 Q1 table |
| 4 | FP cell | Reconstruction — accumulate FP16 error | `err_fp16 += abs(val - f_fp16)` | Part 2 FP table |
| 5 | Training | Zero gradients before backward pass | `optimizer.zero_grad()` | — |
| 6 | Training | Backpropagate loss | `loss.backward()` | — |
| 7 | MX FP loop | Set scale bits for MX FP specs | `mxfp_specs['scale_bits'] = 8` | Part 2 Sec 2.4c |
| 8 | MX INT loop | Set block size in MX INT specs | `mxint_specs['block_size'] = blocksize` | Part 2 Sec 2.4b |
| 9 | Eval cell | Return accuracy from `test_model_mx()` | `return correct / total` | Part 2 Sec 2.5 |


## PART 1: ENVIRONMENT SETUP AND QUANTIZE FP32

First, we'll set up our environment by installing necessary libraries and checking available hardware.

Install specific versions of PyTorch, NumPy, and torchaudio/vision to ensure compatibility.
  
This ensures consistent results across different systems and environments

In [ ]:
# Install compatible libraries
!pip install torch torchvision torchaudio
!pip install 'torch==2.10.0' --quiet


### Check the version of numpy and pytorch

Check the versions of installed libraries and GPU availability.  
Verifies that the correct software stack and hardware resources are ready

In [ ]:
import torch
import numpy as np
import torch.quantization as quant

# Check Numpy, Pytorch version and available hardware
print("Numpy version:", np.__version__)
print("Pytorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())


It should show something like:


```
Numpy version: 2.0.2
Pytorch version: 2.10.0+cu128
CUDA version: 12.8
GPU available: True   ← True only if GPU runtime is selected
```
If numpy shows 2.x instead of 1.26.4:
- Click **Runtime → Restart session**, then re-run from the top.


In [ ]:
!pip install git+https://github.com/liuhao-97/microxcaling.git --no-deps --quiet

### Quantize from FP32 to int8, int6, int4

Define functions to quantize float32 to int8, int6, int4 and reconstruct back.  
Includes MSE computation and bit-level binary formatting.  


#### Takeaway: Shows the effect of reducing precision on numeric accuracy.  

In [ ]:
import numpy as np

def quantize(x, bits):
    """Directly quantize float values to integer range without any scale factor.
    Values outside the integer range are clipped."""
    if bits < 2:
        raise ValueError('Bits must be >= 2')
    qmax =  2**(bits - 1) - 1
    qmin = -2**(bits - 1)
    q = np.round(x).astype(int)
    q = np.clip(q, qmin, qmax)
    return q

def int_to_bin(x, bits):
    """Convert signed integer to two's complement binary string."""
    qmin = -2**(bits - 1)
    qmax =  2**(bits - 1) - 1
    x = np.clip(x, qmin, qmax)
    if x < 0:
        x = (1 << bits) + x
    return format(x, f'0{bits}b')

fp32_values = np.array([-0.3006, -12.2760,  0.7851,  6.6808, -2.2207,
                          0.9973, -10.8371, 13.1378,  1.3044,  5.2754], dtype=np.float32)

for bits in [8, 6, 4]:
    print(f'\n==== Quantizing to INT{bits} ====')
    # To Do 1: Call quantize() on fp32_values with the current bit-width.
    # Hint: q_vals = quantize(fp32_values, bits)
    q_vals = None

    # Cast back to float for error calculation (no scale — compare int vs original float)
    q_float = q_vals.astype(np.float32)

    # To Do 2: Compute MAE between original FP32 and the clipped integer values.
    # Hint: mae = np.mean(np.abs(fp32_values - q_float))
    mae = None

    # To Do 3: Compute MSE between original FP32 and the clipped integer values.
    # Hint: mse = np.mean((fp32_values - q_float) ** 2)
    mse = None

    print(f"{'FP32':>10} {'INT':>6} {'BINARY':>{bits+2}}")
    for f, q in zip(fp32_values, q_vals):
        binary = int_to_bin(q, bits)
        print(f'{f:10.4f} {q:6} {binary:>{bits+2}}')
    print(f'Quantization MAE: {mae:.6f}')
    print(f'Quantization MSE: {mse:.6f}')
    print('Note: error includes both rounding loss AND clipping loss for out-of-range values.')

### Quantize from FP32 to FP16, BF16, FP8_E4M3, FP8_E5M2

Utility functions for converting between FP32 and various formats (FP16, BF16, FP8)     
Takeaway: Understand IEEE754-style bit representations for reduced-precision floats

In [ ]:
import numpy as np
import struct

# ==== Utility Functions to Convert Between Float and Bit Representations ====

# Convert float32 to 32-bit integer representation
def float32_to_bits(f):
    return struct.unpack('>I', struct.pack('>f', f))[0]

# Convert 32-bit integer bits back to float32
def bits_to_float32(b):
    return struct.unpack('>f', struct.pack('>I', b))[0]

# Convert float32 to IEEE 754 half-precision (fp16) 16-bit binary
def float32_to_fp16(f):
    return np.float16(f).view(np.uint16)

# Convert fp16 binary back to float32
def fp16_to_float32(bits):
    return np.float16(bits.view(np.float16)).astype(np.float32)

# Convert float32 to bfloat16 (high 16 bits of float32)
def float32_to_bf16(f):
    bits = float32_to_bits(f)
    return bits >> 16

# Convert bfloat16 (high 16 bits) back to float32
def bf16_to_float32(bf16_bits):
    return bits_to_float32(bf16_bits << 16)

# ==== Manual FP8 Conversions ====

# Convert float32 to FP8 E4M3 (1 sign bit, 4 exponent bits, 3 mantissa bits)
def float32_to_fp8_e4m3(f):
    f_bits = float32_to_bits(f)
    sign = (f_bits >> 31) & 0x1
    exponent = (f_bits >> 23) & 0xFF
    mantissa = f_bits & 0x7FFFFF

    fp32_bias = 127
    fp8_bias = 7
    exp_unbiased = exponent - fp32_bias + fp8_bias

    if exponent == 0 or exp_unbiased <= 0:
        return sign << 7  # zero
    elif exp_unbiased >= 0b1111:
        return (sign << 7) | (0b1111 << 3)  # inf/nan
    else:
        mant = mantissa >> (23 - 3)
        return (sign << 7) | (exp_unbiased << 3) | (mant & 0b111)

# Convert FP8 E4M3 8-bit value back to float32
def fp8_e4m3_to_float32(fp8):
    sign = (fp8 >> 7) & 0x1
    exponent = (fp8 >> 3) & 0xF
    mantissa = fp8 & 0b111

    if exponent == 0 and mantissa == 0:
        return -0.0 if sign else 0.0
    elif exponent == 0b1111:
        return float('-inf') if sign else float('inf')

    fp8_bias = 7
    fp32_bias = 127
    exp = exponent - fp8_bias + fp32_bias
    mant = mantissa << (23 - 3)
    bits = (sign << 31) | (exp << 23) | mant
    return bits_to_float32(bits)

# Convert float32 to FP8 E5M2 (1 sign bit, 5 exponent bits, 2 mantissa bits)
def float32_to_fp8_e5m2(f):
    f_bits = float32_to_bits(f)
    sign = (f_bits >> 31) & 0x1
    exponent = (f_bits >> 23) & 0xFF
    mantissa = f_bits & 0x7FFFFF

    fp32_bias = 127
    fp8_bias = 15
    exp_unbiased = exponent - fp32_bias + fp8_bias

    if exponent == 0 or exp_unbiased <= 0:
        return sign << 7  # zero
    elif exp_unbiased >= 0b11111:
        return (sign << 7) | (0b11111 << 2)  # inf/nan
    else:
        mant = mantissa >> (23 - 2)
        return (sign << 7) | (exp_unbiased << 2) | (mant & 0b11)

# Convert FP8 E5M2 8-bit value back to float32
def fp8_e5m2_to_float32(fp8):
    sign = (fp8 >> 7) & 0x1
    exponent = (fp8 >> 2) & 0x1F
    mantissa = fp8 & 0b11

    if exponent == 0 and mantissa == 0:
        return -0.0 if sign else 0.0
    elif exponent == 0b11111:
        return float('-inf') if sign else float('inf')

    fp8_bias = 15
    fp32_bias = 127
    exp = exponent - fp8_bias + fp32_bias
    mant = mantissa << (23 - 2)
    bits = (sign << 31) | (exp << 23) | mant
    return bits_to_float32(bits)



### Run the conversion on sample FP32 values
Run the conversion on sample FP32 values and display their bit-level layout. Helpful for visualizing how exponent and mantissa bits change across formats

In [ ]:
# Sample values
# ==== Run Example on Sample Float Values ====
values = [-0.3006, -12.2760,   0.7851,   6.6808,  -2.2207,   0.9973, -10.8371,
          13.1378,   1.3044,   5.2754]

# Print original and quantized bit representations
# Print header
print(f"{'Value':>8} | {'FP32 (S|E|M)':>32} | {'FP16 (S|E|M)':>20} | {'BF16 (S|E|M)':>20} | {'FP8-E4M3 (S|E|M)':>20} | {'FP8-E5M2 (S|E|M)':>20}")
print("-"*120)
for val in values:
    fp32_bits = float32_to_bits(val)
    fp16_bits = float32_to_fp16(val)
    bf16_bits = float32_to_bf16(val)
    fp8_e4m3 = float32_to_fp8_e4m3(val)
    fp8_e5m2 = float32_to_fp8_e5m2(val)

    # Extract sign, exponent, mantissa bits
    fp32_s = (fp32_bits >> 31) & 0x1
    fp32_e = (fp32_bits >> 23) & 0xFF
    fp32_m = fp32_bits & 0x7FFFFF

    fp16_s = (fp16_bits >> 15) & 0x1
    fp16_e = (fp16_bits >> 10) & 0x1F
    fp16_m = fp16_bits & 0x3FF

    bf16_s = (bf16_bits >> 15) & 0x1
    bf16_e = (bf16_bits >> 7) & 0xFF
    bf16_m = bf16_bits & 0x7F

    fp8_4_s = (fp8_e4m3 >> 7) & 0x1
    fp8_4_e = (fp8_e4m3 >> 3) & 0xF
    fp8_4_m = fp8_e4m3 & 0x7

    fp8_5_s = (fp8_e5m2 >> 7) & 0x1
    fp8_5_e = (fp8_e5m2 >> 2) & 0x1F
    fp8_5_m = fp8_e5m2 & 0x3

    print(f"{val:8.4f} | "
          f"{fp32_s:1b} {fp32_e:08b} {fp32_m:023b} | "
          f"{fp16_s:1b} {fp16_e:05b} {fp16_m:010b} | "
          f"{bf16_s:1b} {bf16_e:08b} {bf16_m:07b} | "
          f"{fp8_4_s:1b} {fp8_4_e:04b} {fp8_4_m:03b}       | "
          f"{fp8_5_s:1b} {fp8_5_e:05b} {fp8_5_m:02b}")

### Convert reduced-precision values back to FP32
Convert reduced-precision values back to FP32 and compare them
Takeaway: Observe how rounding and truncation impact numerical accuracy

In [ ]:
# Print reconstruction results and compute average error
print('\nReconstruction to FP32:')
print(f"{'Value':>8} | {'From FP16':>10} | {'From BF16':>10} | {'From FP8_E4M3':>10} | {'From FP8_E5M2':>10}")
print('-'*60)

err_fp16 = 0
err_bf16 = 0
err_e4m3 = 0
err_e5m2 = 0

for val in values:
    fp16_bits = float32_to_fp16(val)
    bf16_bits = float32_to_bf16(val)
    fp8_e4m3 = float32_to_fp8_e4m3(val)
    fp8_e5m2 = float32_to_fp8_e5m2(val)

    f_fp16 = fp16_to_float32(fp16_bits)
    f_bf16 = bf16_to_float32(bf16_bits)
    f_e4m3 = fp8_e4m3_to_float32(fp8_e4m3)
    f_e5m2 = fp8_e5m2_to_float32(fp8_e5m2)

    print(f'{val:8.4f} | {f_fp16:10.4f} | {f_bf16:10.4f} | {f_e4m3:10.4f} | {f_e5m2:10.4f}')

    # To Do 5: Accumulate the absolute error for FP16.
    # The same pattern applies to bf16, e4m3, e5m2 — those are already done for you.
    # Hint: err_fp16 += abs(val - f_fp16)
    err_fp16 += None
    err_bf16 += abs(val - f_bf16)
    err_e4m3 += abs(val - f_e4m3)
    err_e5m2 += abs(val - f_e5m2)

n = len(values)
print('\nAverage Absolute Errors:')
print(f"{'FP16':>10}: {err_fp16 / n:.6f}")
print(f"{'BF16':>10}: {err_bf16 / n:.6f}")
print(f"{'FP8-E4M3':>10}: {err_e4m3 / n:.6f}")
print(f"{'FP8-E5M2':>10}: {err_e5m2 / n:.6f}")


## PART 2: Exploring Microscaling (MX) data formats for DNN

This notebook is Part 2 of the Arithmetic and Number Representations lab.

1. Train a Simple MNIST Linear Model and obtain the original accuracy.
2. Quantize and evaluate the output matrix with different MX data formats.
3. Quantize and evaluate the matrix multiplication accumulation with different MX data formats.
4. Evaluate quantized DNN with different MX data formats.



### Train a Simple MNIST Linear Model and obtain the original accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load and preprocess MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 2. Define the model
class LinearModel(nn.Module):
    def __init__(self):
        super(LinearModel, self).__init__()
        self.linear = nn.Linear(28 * 28, 10,bias=True)  # 784 inputs (flattened 28x28), 10 outputs

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten the 28x28 images
        return self.linear(x)

model = LinearModel().to(device)

# 3. Set loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 4. Train the model
def train_model(num_epochs=5):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # To Do 6: Zero out gradients before each backward pass.
            # Hint: optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # To Do 7: Compute gradients via backpropagation.
            # Hint: loss.backward()
            None  # replace with loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 5. Evaluate the model
def test_model():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")

# Run training and testing
train_model(num_epochs=5)
test_model()

In [ ]:
model.eval()  # Set model to evaluation mode
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

with torch.no_grad():
    for i, (images, labels) in enumerate(test_loader):
      if i>=1:
        break
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      print(f"Sample {i+1}:")
      print("Label:", labels)
      print("Outputs:", outputs)
      print("Predicted:", predicted)
      print(" ")

In [ ]:
# Assuming `model` is the trained LinearModel from the previous code
model.eval()  # Set model to evaluation mode

# 1. Extract weight matrix and bias
weight_matrix = model.linear.weight  # Shape: (10, 784)
bias_vector = model.linear.bias      # Shape: (10,)

# 2. Convert to NumPy (optional)
weight_matrix_np = weight_matrix.detach().cpu().numpy()  # Convert to NumPy array
bias_vector_np = bias_vector.detach().cpu().numpy()      # Convert to NumPy array
# 3. Print shapes and example values
print("Weight matrix shape:", weight_matrix.shape)
print("Bias vector shape:", bias_vector.shape)
print("First row of weight matrix (weights for class 0):\n", weight_matrix_np[0, :10])  # First 10 weights for class 0
print("Bias vector:\n", bias_vector_np)
print(" ")

# Process first samples
with torch.no_grad():
    for i, (images, labels) in enumerate(test_loader):
        if i >= 1:
            break
        images, labels = images.to(device), labels.to(device)

        # Flatten images for torch.nn.functional.linear
        images_flat = images.view(-1, 28*28)  # Shape: (1, 784)

        # Compute output using torch.nn.functional.linear
        outputs = torch.nn.functional.linear(input=images_flat, weight=weight_matrix, bias=bias_vector)

        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)

        # Print results in the same format as the second code
        print(f"Sample {i+1}:")
        print("Label:", labels)
        print("Outputs:", outputs)
        print("Predicted:", predicted)
        print(" ")


### Quantize and evaluate the output matrix with different MX data formats.

In [ ]:
import mx
from mx import MxSpecs, linear
from mx.specs import finalize_mx_specs
from mx.mx_ops import _quantize_mx
import torch
import numpy as np
import matplotlib.pyplot as plt
# Removed: pytest (not needed), mx_mapping, matmul, Linear (unused)


In [ ]:
# Set random seed for reproducibility
np.random.seed(43)
torch.manual_seed(43)

for i, (images, labels) in enumerate(test_loader):
    if i == 0:
      images, labels = images.to(device), labels.to(device)
      # Flatten images for torch.nn.functional.linear
      images_flat = images.view(-1, 28*28)  # Shape: (1, 784)
      A = images_flat.detach().cpu().numpy()  # Shape: (1, 784)
      break

# print(A.shape)
B = weight_matrix_np
C = bias_vector_np

# print(B.shape) # Shape:(10, 784)
# print(C.shape) # Shape:(10,)

# Convert to PyTorch tensors
A_torch = torch.from_numpy(A) # Shape: (1, 784)
B_torch = torch.from_numpy(B) # Shape:(10, 784)
C_torch = torch.from_numpy(C) # Shape:(10,)

# Ground truth with NumPy (FP32)
with torch.no_grad():
  D_torch = torch.nn.functional.linear(input=images_flat, weight=weight_matrix, bias=bias_vector)
D_ground_truth = D_torch.detach().cpu().numpy()
print("D_ground_truth: \n",D_ground_truth)
print("D_torch: \n",D_ground_truth)

_, predicted = torch.max(D_torch.data, 1)
print("Label:", labels)
print("Outputs:", outputs)
print("Predicted:", predicted)

### Quantize and evaluate the output matrix with different MX data formats.

In [ ]:
quant_list = ['int8', 'int4', 'int2', 'fp8_e5m2', 'fp8_e4m3', 'fp6_e3m2', 'fp6_e2m3', 'fp4_e2m1', 'fp16', 'bf16']
mae_list = []
mse_list = []

print("Quantization Error Metrics:")
print(f"{'Method':<12} {'MAE':>12} {'MSE':>12}")
print("-" * 36)

for quant_method in quant_list:
    D_mx = _quantize_mx(D_torch, 8, quant_method,
                        block_size=8,
                        axes=[-1],
                        round='nearest',
                        flush_fp32_subnorms=False,
                        custom_cuda=False)

    _, mx_predicted = torch.max(D_mx.data, 1)
    print(f"mx_{quant_method} Truth Label: {labels}, Predicted: {mx_predicted}")

    mae = torch.mean(torch.abs(D_mx - D_torch)).item()
    mse = torch.mean((D_mx - D_torch) ** 2).item()

    mae_list.append(mae)
    mse_list.append(mse)

    print(f"mx{quant_method:<12} {mae:12.6f} {mse:12.6f}")
    print()

# Visualize the metrics
plt.figure(figsize=(14, 6))

# MAE
plt.subplot(1, 2, 1)
plt.bar(quant_list, mae_list)
plt.title('MAE Comparison')
plt.grid(axis='y')
plt.yscale('log')
plt.xticks(
    ticks=range(len(quant_list)),
    labels=[f"mx_{q}" for q in quant_list],
    rotation=90
)

# MSE
plt.subplot(1, 2, 2)
plt.bar(quant_list, mse_list)
plt.title('MSE Comparison')
plt.grid(axis='y')
plt.yscale('log')
plt.xticks(
    ticks=range(len(quant_list)),
    labels=[f"mx_{q}" for q in quant_list],
    rotation=90
)

plt.tight_layout()
plt.show()

### Compare MX_int8 with int8 matrix.

In [ ]:
D_mxint8 = _quantize_mx(D_torch, 8, 'int8',
                block_size=8,
                axes=[-1],
                round='nearest',
                flush_fp32_subnorms=False,
                custom_cuda=False)

_, mxint8_predicted = torch.max(D_mxint8.data, 1)
print(f"mxint8 Truth Label: {labels}, Predicted: {mxint8_predicted}")
print("D_mxint8:\n", D_mxint8 )
print("\n")


mae_mxint8=torch.mean(torch.abs(D_mxint8 - D_torch)).item()
mse_mxint8=torch.mean((D_mxint8 - D_torch) ** 2).item()
print("MAE_mxint8: ",mae_mxint8)
print("MSE_mxint8: ",mse_mxint8)




D_int8 = D_torch.type(torch.int8)
_, int8_predicted = torch.max(D_mxint8.data, 1)
print(f"int8 Truth Label: {labels}, Predicted: {int8_predicted}")
print("D_int8:\n", D_int8)
print("\n")


mae_int8=torch.mean(torch.abs(D_int8 - D_torch)).item()
mse_int8=torch.mean((D_int8 - D_torch) ** 2).item()
print("MAE_int8: ", mae_int8)
print("MSE_int8: ", mse_int8)


# Visualize the metrics
plt.figure(figsize=(12, 6))

# MAE
plt.subplot(1, 2, 1)
plt.bar(['mxint8', 'int8'],[mae_mxint8,mae_int8])
plt.title('MAE Comparison')
plt.grid(axis='y')
plt.yscale('log')

# Training time comparison
plt.subplot(1, 2, 2)
plt.bar(['mxint8', 'int8'],[mse_mxint8,mse_int8])
plt.title('MSE Comparison')
plt.grid(axis='y')
plt.yscale('log')

plt.tight_layout()
plt.show()

### Test MX_int8 for various block sizes
Plot and analyze how error trends vary with block size

In [ ]:
blocksize_list = [1,2,4,6,8,10]
mae_list = []
mse_list = []

for blocksize in blocksize_list:
  D_mx = _quantize_mx(D_torch, 32, 'int8',
                  block_size=blocksize,
                  axes=[-1],
                  round='nearest',
                  flush_fp32_subnorms=False,
                  custom_cuda=False)
  _, mx_predicted = torch.max(D_mx.data, 1)
  print(f"mxint8 blocksize: {blocksize}, Truth Label: {labels}, Predicted: {mx_predicted}")
  # print("D_mx:\n", D_mx)
  print("\n")

  mae_list.append(torch.mean(torch.abs(D_mx - D_torch)).item())
  mse_list.append(torch.mean((D_mx - D_torch) ** 2).item())
  print(f"blocksize {blocksize}: ")
  print(f"MAE: {torch.mean(torch.abs(D_mx - D_torch))}")
  print(f"MSE: {torch.mean((D_mx - D_torch) ** 2)}\n")

# Visualize the metrics
plt.figure(figsize=(14, 6))

# MAE
plt.subplot(1, 2, 1)
plt.plot(blocksize_list,mae_list,markersize=10, marker='*')
plt.xlabel('block size', size=20)
plt.ylabel('MAE', size=20)
plt.title('MAE Comparison')
# plt.ylim(0.06, 0.067)  # Adjust as needed
plt.grid(axis='y')

# Training time comparison
plt.subplot(1, 2, 2)
plt.plot(blocksize_list,mse_list,markersize=10, marker='*')
plt.title('MSE Comparison')
plt.xlabel('block size', size=20)
plt.ylabel('MSE', size=20)
# plt.ylim(0.005, 0.0062)  # Adjust as needed
plt.grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Define helper function for matrix multiplication accumulation
def matmul_accumulate(A, B, C, mx_specs=None):
    if mx_specs is None:
        # Native PyTorch FP32
        # return torch.matmul(A, B) + C
        return torch.nn.functional.linear(A, B, C)
    else:
        # Use microxcaling's linear function for quantized computation
        return mx.linear(A, B, C, mx_specs=mx_specs)

# Compute accuracy metrics
def compute_errors(ground_truth, result):
    result_np = result.cpu().numpy()  # Move back to CPU for comparison
    abs_error = np.abs(ground_truth - result_np)
    mae = np.mean(abs_error)
    maxae = np.max(abs_error)
    return mae, maxae

In [ ]:
# Run with different data types
# FP32 (no quantization)
fp32_result = matmul_accumulate(A_torch, B_torch, C_torch, mx_specs=None)

fp32_mae, fp32_maxae = compute_errors(D_ground_truth, fp32_result)
print("Accuracy Comparison:")
print(f"FP32 - MAE: {fp32_mae:.6f}, MaxAE: {fp32_maxae:.6f}")

### Test mx_int quantization (mx_int8/mx_int4/mx_int2) for various block sizes
   
Plot and analyze how error trends vary with block size

In [ ]:
# Visualize the metrics
plt.figure(figsize=(14, 6))
quant_list = ['int8', 'int4', 'int2']
blocksize_list = [2,4,8,12,16,24,32,40,48,56,64]
for quant_method in quant_list:
  mae_list=[]
  mse_list=[]
  for blocksize in blocksize_list:
    mxint_specs=MxSpecs()
    mxint_specs['w_elem_format'] = quant_method
    mxint_specs['a_elem_format'] = quant_method
    # To Do: set the block size for MX quantization.
    # Hint: mxint_specs['block_size'] = blocksize
    mxint_specs['block_size'] = None
    mxint_specs['shared_exp_method'] = 'max'

    mxint_result = matmul_accumulate(A_torch.to(device), B_torch.to(device), C_torch.to(device), mx_specs=mxint_specs)
    mae_list.append(torch.mean(torch.abs(mxint_result - D_torch)).cpu())
    mse_list.append(torch.mean((mxint_result - D_torch) ** 2).cpu())

    print(f"mx_{quant_method} blocksize {blocksize}")
    mae = torch.mean(torch.abs(mxint_result - D_torch)).item()
    mse = torch.mean((mxint_result - D_torch) ** 2).item()
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}\n")

    _, mxint_predicted = torch.max(mxint_result.data, 1)
    # print(f"mx{quant_method} blocksize: {blocksize}, Truth Label: {labels}, Predicted: {mxint_predicted}")
    # print(f"mx{quant_method}_result:\n", mxint_result)
    print("\n")

  # MAE
  plt.subplot(1,2,1)
  plt.plot(blocksize_list,mae_list,markersize=10, marker='*', label=quant_method)
  plt.xlabel('block size', size=20)
  plt.ylabel('MAE', size=20)
  plt.title('MAE Comparison')
  plt.grid(axis='y')

  plt.subplot(1,2,2)
  plt.plot(blocksize_list,mse_list,markersize=10, marker='*', label=quant_method)
  plt.xlabel('block size', size=20)
  plt.ylabel('MSE', size=20)
  plt.title('MSE Comparison')
  plt.grid(axis='y')

plt.legend(fontsize=20)
plt.tight_layout()
plt.show()

### Test mx_fp quantization mx_(fp8_e5m2, fp8_e4m3, fp6_e3m2, fp6_e2m3, fp4_e2m1,  fp16, bf16) for various block sizes
   
Plot and analyze how error trends vary with block size

In [ ]:
# Visualize the metrics
plt.figure(figsize=(14, 6))
quant_list = ['fp8_e5m2', 'fp8_e4m3', 'fp6_e3m2', 'fp6_e2m3', 'fp4_e2m1',  'fp16', 'bf16']

blocksize_list = [4, 8,16,24,32,40,48,56,64,]
for quant_method in quant_list:
  mae_list=[]
  mse_list=[]
  for blocksize in blocksize_list:
    mxfp_specs=MxSpecs()
    # To Do 8: Set the number of scale bits for MX floating-point quantization.
    # Hint: mxfp_specs['scale_bits'] = 8
    mxfp_specs['scale_bits'] = None
    mxfp_specs['w_elem_format'] = quant_method
    mxfp_specs['a_elem_format'] = quant_method
    mxfp_specs['block_size'] = blocksize
    mxfp_specs['shared_exp_method'] = 'max'
    mxfp_specs = finalize_mx_specs(mxfp_specs)

    mxfp_result = matmul_accumulate(A_torch, B_torch, C_torch, mx_specs=mxfp_specs).to(device)
    mae_list.append(torch.mean(torch.abs(mxfp_result - D_torch)).cpu())
    mse_list.append(torch.mean((mxfp_result - D_torch) ** 2).cpu())

    print(f"mx_{quant_method} blocksize {blocksize}")
    print(f"MAE: {torch.mean(torch.abs(mxfp_result - D_torch))}")
    print(f"MSE: {torch.mean((mxfp_result - D_torch) ** 2)}\n")

    _, mxfp_predicted = torch.max(mxfp_result.data, 1)
    # print(f"mx{quant_method} blocksize: {blocksize}, Truth Label: {labels}, Predicted: {mxfp_predicted}")
    # print(f"mx{quant_method}_result:\n", mxfp_result)
    print("\n")

  # MAE
  plt.subplot(1,2,1)
  plt.plot(blocksize_list,mae_list,markersize=10, marker='*', label=quant_method)
  plt.title('MAE Comparison')
  plt.grid(axis='y')

  plt.subplot(1,2,2)
  plt.plot(blocksize_list,mse_list,markersize=10, marker='*', label=quant_method)
  plt.title('MSE Comparison')
  plt.grid(axis='y')

# plt.grid(axis='y')
plt.legend(fontsize=20)
plt.tight_layout()
plt.show()

### Evaluate quantized DNN with different MX data formats.




In [ ]:
# Evaluate the model
def test_model():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f"Original Test Accuracy: {accuracy:.2f}")
    return accuracy


def test_model_mx(mx_specs):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
      for i, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)

        # Flatten images for torch.nn.functional.linear
        images_flat = images.view(-1, 28*28).cpu()  # Shape: (1, 784)

        # Compute output using torch.nn.functional.linear
        # outputs = torch.nn.functional.linear(input=images_flat, weight=weight_matrix, bias=bias_vector)
        outputs = mx.linear(images_flat, B_torch.cpu(), C_torch.cpu(), mx_specs=mx_specs)
        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.cpu()).sum().item()
  # To Do: compute and return accuracy as fraction of correct predictions.
  # Hint: return correct / total
  accuracy = None
  return accuracy

# Re-evaluate model accuracy using original (full-precision) weights
# This is the baseline for assessing quantized inference accuracy
original_accuracy = test_model()

# Visualize the metrics
plt.figure(figsize=(14, 6))

quant_list = ['int8', 'int4', 'int2','fp8_e5m2', 'fp8_e4m3', 'fp6_e3m2', 'fp6_e2m3', 'fp4_e2m1',  'fp16', 'bf16']
blocksize_list = [32,256,]

# Define test_model_mx: inference with quantized formats using microxcaling
# Used to simulate low-bit inference performance
for blocksize in blocksize_list:
  accuracy_list=[]
  for quant_method in quant_list:
    mxint_specs=MxSpecs()
    mxint_specs['w_elem_format'] = quant_method
    mxint_specs['a_elem_format'] = quant_method
    mxint_specs['block_size'] = blocksize
    mxint_specs['shared_exp_method'] = 'max'
    accuracy = test_model_mx(mxint_specs)
    accuracy_list.append(accuracy)
    print(f"quant method:mx{quant_method}, blocksize: {blocksize}, accuracy: {accuracy}")

  if blocksize == 32:
    plt.subplot(1,2,1)
    plt.bar(['original_accuracy']+quant_list, [original_accuracy] + accuracy_list, )
    plt.xlabel('quant method', size=20)
    plt.ylabel('accuracy', size=20)
    plt.title(f'Accuracy Comparison (blocksize {blocksize})')
    plt.xticks(rotation=90)  # ← add this
    plt.grid(axis='y')
    # plt.ylim(0.84,1)

  if blocksize == 256:
    plt.subplot(1,2,2)
    plt.bar(['original_accuracy']+quant_list, [original_accuracy] + accuracy_list, )
    plt.xlabel('quant method', size=20)
    plt.ylabel('accuracy', size=20)
    plt.title(f'Accuracy Comparison (blocksize {blocksize})')
    plt.xticks(rotation=90)  # ← add this
    plt.grid(axis='y')
    # plt.ylim(0.7,1)

# plt.legend()  # removed: no labeled bars here

plt.tight_layout()
plt.show()

In [ ]:
labels_x = ['FP32 baseline'] + ['mx_' + q for q in quant_list]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, blocksize in zip(axes, blocksize_list):
    values = [original_accuracy] + results_by_blocksize[blocksize]
    bars = ax.bar(labels_x, values)

    # Add value labels on top of each bar
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', size=8, rotation=90)

    ax.set_title(f'Accuracy Comparison (blocksize={blocksize})', size=13)
    ax.set_xlabel('Quant method', size=12)
    ax.set_ylabel('Accuracy',     size=12)
    ax.set_xticks(range(len(labels_x)))
    ax.set_xticklabels(labels_x, rotation=45, ha='center', size=10)
    ax.set_ylim(0, 1.15)  # slightly taller to fit the number above each bar
    ax.grid(axis='y')

plt.tight_layout()
plt.show()

---
## 🎯 Student Tasks

Each task has two parts:
- **GIVEN** — already works, just run it and observe the output
- **TODO** — you must complete the code using the `# Hint:` comments

Record all results in your worksheet.

---

### ✅ Task 1 — Wide vs Narrow Range: INT, FP, MX INT, MX FP

Two fixed tensors are provided. The INT8 example is done for you.
Complete the rest for INT4, FP8_E4M3, MX_INT8, and MX_FP8_E4M3.

Record all results in **Worksheet Part 3, Task 1 table**:

| Format | Wide MAE | Wide MSE | Narrow MAE | Narrow MSE |
|--------|----------|----------|------------|------------|
| INT8 (given) | | | | |
| INT4 | | | | |
| FP8_E4M3 | | | | |
| MX_INT8 | | | | |
| MX_FP8_E4M3 | | | | |
| MXFP4_E2M1 | | | | |
---

In [ ]:
# ════════════════════════════════════════════════════════════════════
# TASK 1 — Wide vs Narrow Range — DO NOT change the tensors
# Complete the TODO sections using the hints provided.
# ════════════════════════════════════════════════════════════════════

wide_values   = np.array([-95.3,  42.1,  -0.5,  78.9, -33.2,
                             0.3,  66.7, -88.1,  15.4, -50.0], dtype=np.float32)
narrow_values = np.array([ -0.95,  0.42, -0.05,  0.79, -0.33,
                             0.03,  0.67, -0.88,  0.15, -0.50], dtype=np.float32)

# ── GIVEN: INT8 on both tensors ───────────────────────────────────────
print('=== INT8 bare (given) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    q_vals  = quantize(tensor, 8)
    q_float = q_vals.astype(np.float32)   # cast back for error calculation
    mae = np.mean(np.abs(tensor - q_float))
    mse = np.mean((tensor - q_float) ** 2)
    print(f'  {label} → MAE: {mae:.6f}   MSE: {mse:.6f}')

# ── TODO: INT4 on both tensors ────────────────────────────────────────
print('\n=== INT4 bare (TODO) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    # Hint: q_vals = quantize(tensor, 4)
    q_vals  = None
    # Hint: q_float = q_vals.astype(np.float32)
    q_float = None
    # Hint: mae = np.mean(np.abs(tensor - q_float))
    mae = None
    # Hint: mse = np.mean((tensor - q_float) ** 2)
    mse = None
    print(f'  {label} → MAE: {mae:.6f}   MSE: {mse:.6f}')

# ── TODO: FP8_E4M3 on both tensors ───────────────────────────────────
print('\n=== FP8_E4M3 (TODO) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    # Hint: errors = [abs(float(v) - fp8_e4m3_to_float32(float32_to_fp8_e4m3(float(v)))) for v in tensor]
    errors = None
    # Hint: sq_err = [(float(v) - fp8_e4m3_to_float32(float32_to_fp8_e4m3(float(v))))**2 for v in tensor]
    sq_err = None
    print(f'  {label} → MAE: {np.mean(errors):.6f}   MSE: {np.mean(sq_err):.6f}')

# ── TODO: FP4_E2M1 on both tensors ───────────────────────────────────
# NOTE: requires float32_to_fp4_e2m1 and fp4_e2m1_to_float32
#       to be defined in the extra credit section first.
# print('\n=== FP4_E2M1 (TODO) ===')
# for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
#     # Hint: errors = [abs(float(v) - fp4_e2m1_to_float32(float32_to_fp4_e2m1(float(v)))) for v in tensor]
#     errors = None
#     # Hint: sq_err = [(float(v) - fp4_e2m1_to_float32(float32_to_fp4_e2m1(float(v))))**2 for v in tensor]
#     sq_err = None
#     print(f'  {label} → MAE: {np.mean(errors):.6f}   MSE: {np.mean(sq_err):.6f}')

# ── TODO: MX_INT8 blocksize=4 on both tensors ────────────────────────
print('\n=== MX_INT8 blocksize=4 (TODO) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    tensor_t = torch.tensor(tensor).unsqueeze(0)
    fp32_ref = tensor_t.clone()
    specs = MxSpecs()
    # Hint: specs['w_elem_format'] = 'int8'
    specs['w_elem_format']     = None
    # Hint: specs['a_elem_format'] = 'int8'
    specs['a_elem_format']     = None
    # Hint: specs['block_size'] = 4
    specs['block_size']        = None
    specs['shared_exp_method'] = 'max'
    W_eye  = torch.eye(10)
    mx_out = mx.linear(tensor_t, W_eye, None, mx_specs=specs)
    mae = torch.mean(torch.abs(mx_out - fp32_ref)).item()
    mse = torch.mean((mx_out - fp32_ref) ** 2).item()
    print(f'  {label} → MAE: {mae:.6f}   MSE: {mse:.6f}')

# ── TODO: MX_FP8_E4M3 blocksize=4 on both tensors ────────────────────
print('\n=== MX_FP8_E4M3 blocksize=4 (TODO) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    tensor_t = torch.tensor(tensor).unsqueeze(0)
    fp32_ref = tensor_t.clone()
    specs = MxSpecs()
    # Hint: specs['w_elem_format'] = 'fp8_e4m3'
    specs['w_elem_format']     = None
    # Hint: specs['a_elem_format'] = 'fp8_e4m3'
    specs['a_elem_format']     = None
    # Hint: specs['block_size'] = 4
    specs['block_size']        = None
    specs['shared_exp_method'] = 'max'
    W_eye  = torch.eye(10)
    mx_out = mx.linear(tensor_t, W_eye, None, mx_specs=specs)
    mae = torch.mean(torch.abs(mx_out - fp32_ref)).item()
    mse = torch.mean((mx_out - fp32_ref) ** 2).item()
    print(f'  {label} → MAE: {mae:.6f}   MSE: {mse:.6f}')

# ── TODO: MX_FP4_E2M1 blocksize=4 on both tensors ────────────────────
print('\n=== MX_FP4_E2M1 blocksize=4 (TODO) ===')
for label, tensor in [('Wide  ', wide_values), ('Narrow', narrow_values)]:
    tensor_t = torch.tensor(tensor).unsqueeze(0)
    fp32_ref = tensor_t.clone()
    specs = MxSpecs()
    # Hint: specs['w_elem_format'] = 'fp4_e2m1'
    specs['w_elem_format']     = None
    # Hint: specs['a_elem_format'] = 'fp4_e2m1'
    specs['a_elem_format']     = None
    # Hint: specs['block_size'] = 4
    specs['block_size']        = None
    specs['shared_exp_method'] = 'max'
    W_eye  = torch.eye(10)
    mx_out = mx.linear(tensor_t, W_eye, None, mx_specs=specs)
    mae = torch.mean(torch.abs(mx_out - fp32_ref)).item()
    mse = torch.mean((mx_out - fp32_ref) ** 2).item()
    print(f'  {label} → MAE: {mae:.6f}   MSE: {mse:.6f}')

### ✅ Task 2 — MX vs FP32 Baseline: MAE and MSE

A fixed sample tensor is provided. MX_INT8 is done for you.
Complete the rest for INT4, INT2, FP8_E4M3, FP4_E2M1.

Record results in **Worksheet Part 3, Task 2 table**:

| Format | MAE vs FP32 | MSE vs FP32 |
|--------|-------------|-------------|
| MX_INT8 (given) | | |
| MX_INT4 | | |
| MX_INT2 | | |
| MX_FP8_E4M3 | | |
| MX_FP4_E2M1 | | |


---

In [ ]:
# ════════════════════════════════════════════════════════════════════
# TASK 2 — MX vs FP32 Baseline
# DO NOT change the sample tensor or the GIVEN section.
# Complete the TODO sections using the hints provided.
# ════════════════════════════════════════════════════════════════════

torch.manual_seed(42)
sample_A    = torch.randn(1, 784, dtype=torch.float32)   # fixed for everyone
fp32_result = torch.nn.functional.linear(sample_A, B_torch.cpu(), C_torch.cpu())

print(f"{'Format':<22} {'MAE vs FP32':>14} {'MSE vs FP32':>14}")
print('-' * 52)

# ── GIVEN: MX_INT8 ───────────────────────────────────────────────────
specs = MxSpecs()
specs['w_elem_format']     = 'int8'
specs['a_elem_format']     = 'int8'
specs['block_size']        = 16
specs['shared_exp_method'] = 'max'
mx_result = mx.linear(sample_A, B_torch.cpu(), C_torch.cpu(), mx_specs=specs)
mae = torch.mean(torch.abs(mx_result - fp32_result)).item()
mse = torch.mean((mx_result - fp32_result) ** 2).item()
print(f"{'mx_int8 (given)':<22} {mae:>14.6f} {mse:>14.6f}")

# ── TODO: complete for the remaining formats ─────────────────────────
for fmt_name, fmt_str in [('mx_int4',     'int4'),
                           ('mx_int2',     'int2'),
                           ('mx_fp8_e4m3', 'fp8_e4m3'),
                           ('mx_fp4_e2m1', 'fp4_e2m1')]:
    specs = MxSpecs()
    # Hint: specs['w_elem_format'] = fmt_str
    specs['w_elem_format']     = None
    # Hint: specs['a_elem_format'] = fmt_str
    specs['a_elem_format']     = None
    # Hint: specs['block_size'] = 16
    specs['block_size']        = None
    specs['shared_exp_method'] = 'max'
    # Hint: mx_result = mx.linear(sample_A, B_torch.cpu(), C_torch.cpu(), mx_specs=specs)
    mx_result = None
    # Hint: mae = torch.mean(torch.abs(mx_result - fp32_result)).item()
    mae = None
    # Hint: mse = torch.mean((mx_result - fp32_result) ** 2).item()
    mse = None
    print(f'{fmt_name:<22} {mae:>14.6f} {mse:>14.6f}')


### ✅ Task 3 — Full Model Accuracy at Fixed Block Sizes

MX_INT8 at blocksize=8 is done for you.
Complete the rest for INT4, INT2, FP8_E4M3, FP4_E2M1 across blocksizes 8, 32, 128.

Record results in **Worksheet Part 2, Task 3 table**:

| Format | blocksize=8 | blocksize=32 | blocksize=128 |
|--------|-------------|--------------|---------------|
| MX_INT8 (given) | | | |
| MX_INT4 | | | |
| MX_INT2 | | | |
| MX_FP8_E4M3 | | | |
| MX_FP4_E2M1 | | | |

📝 **Q**: At which block size does accuracy drop most for INT2 and FP4? Does a larger block size always help?

In [ ]:
# ════════════════════════════════════════════════════════════════════
# TASK 3 — Full Model Accuracy at Fixed Block Sizes
# DO NOT change formats or block sizes.
# Complete the TODO section using the hints provided.
# ════════════════════════════════════════════════════════════════════

fixed_formats    = ['int8', 'int4', 'int2', 'fp8_e4m3', 'fp4_e2m1']
fixed_blocksizes = [8, 32, 128]

print(f"{'Format':<22} {'bs=8':>10} {'bs=32':>10} {'bs=128':>10}")
print('-' * 56)

# ── GIVEN: MX_INT8 across all block sizes ────────────────────────────
row = f"{'mx_int8 (given)':<22}"
for blocksize in fixed_blocksizes:
    specs = MxSpecs()
    specs['w_elem_format']     = 'int8'
    specs['a_elem_format']     = 'int8'
    specs['block_size']        = blocksize
    specs['shared_exp_method'] = 'max'
    acc = test_model_mx(specs)
    row += f'{acc:>10.4f}'
print(row)

# ── TODO: complete for the remaining formats ─────────────────────────
for quant_method in ['int4', 'int2', 'fp8_e4m3', 'fp4_e2m1']:
    row = f'mx_{quant_method:<19}'
    for blocksize in fixed_blocksizes:
        specs = MxSpecs()
        # Hint: specs['w_elem_format'] = quant_method
        specs['w_elem_format']     = None
        # Hint: specs['a_elem_format'] = quant_method
        specs['a_elem_format']     = None
        # Hint: specs['block_size'] = blocksize
        specs['block_size']        = None
        specs['shared_exp_method'] = 'max'
        # Hint: acc = test_model_mx(specs)
        acc = None
        row += f'{acc:>10.4f}'
    print(row)


---
### ⭐ Extra Credit — Implement FP4 E2M1 From Scratch

You have already seen FP8 E4M3 implemented in Part 1.
FP4 E2M1 follows the **exact same structure** — only the
number of bits and bias value change.

Use the FP8 E4M3 functions as your template:

| Format       | Sign | Exponent bits | Mantissa bits | Bias |
|--------------|------|---------------|---------------|------|
| FP8 E4M3     | 1    | 4             | 3             | 7    |
| **FP4 E2M1** | **1**| **2**         | **1**         | **1**|

### What changes from FP8 E4M3 → FP4 E2M1:
- `fp4_bias = 1` instead of `fp8_bias = 7`
- Exponent field is **2 bits**, mask `0x3` instead of `0xF`
- Mantissa field is **1 bit**, mask `0x1` instead of `0x7`
- Shift mantissa by `(23 - 1)` instead of `(23 - 3)`
- Sign bit is at position 3: `sign << 3` instead of `sign << 7`
- Overflow threshold is `0b11` instead of `0b1111`

### Representable positive values (only 8):
`0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0`

### Your task:
Implement `float32_to_fp4_e2m1` and `fp4_e2m1_to_float32`
following the FP8 E4M3 example below.
Once complete, the FP4 rows in **Task 1** will automatically run.

📝 **Worksheet Part 3**: Record your FP4 errors after rerunning task 1 with FP4 call

In [ ]:
# ════════════════════════════════════════════════════════════════════
# EXTRA CREDIT — Implement FP4 E2M1
# ════════════════════════════════════════════════════════════════════

# ── EXAMPLE: FP8 E4M3 (already implemented — use this as your guide)
# def float32_to_fp8_e4m3(f):
#     f_bits    = float32_to_bits(f)
#     sign      = (f_bits >> 31) & 0x1
#     exp       = (f_bits >> 23) & 0xFF
#     mant      = f_bits & 0x7FFFFF
#     fp32_bias = 127
#     fp8_bias  = 7                        ← change to 1 for FP4
#     exp_fp8   = exp - fp32_bias + fp8_bias
#     if exp == 0 or exp_fp8 <= 0:
#         return sign << 7                 ← change to sign << 3 for FP4
#     elif exp_fp8 >= 0b1111:              ← change to 0b11 for FP4
#         return (sign << 7) | (0b1111 << 3)
#     else:
#         mant_bits = mant >> (23 - 3)     ← change (23-3) to (23-1) for FP4
#         return (sign << 7) | (exp_fp8 << 3) | (mant_bits & 0b111)
#
# def fp8_e4m3_to_float32(fp8):
#     sign = (fp8 >> 7) & 0x1
#     exp  = (fp8 >> 3) & 0xF             ← change to >> 1 and & 0x3 for FP4
#     mant = fp8 & 0x7                    ← change to & 0x1 for FP4
#     if exp == 0 and mant == 0:
#         return -0.0 if sign else 0.0
#     elif exp == 0b1111:                  ← change to 0b11 for FP4
#         return float('-inf') if sign else float('inf')
#     fp8_bias  = 7                        ← change to 1 for FP4
#     fp32_bias = 127
#     exp32  = exp - fp8_bias + fp32_bias
#     mant32 = mant << (23 - 3)           ← change (23-3) to (23-1) for FP4
#     bits   = (sign << 31) | (exp32 << 23) | mant32
#     return bits_to_float32(bits)

# ── YOUR IMPLEMENTATION ───────────────────────────────────────────────
def float32_to_fp4_e2m1(f):
    # TODO: implement following the FP8 E4M3 example above
    pass

def fp4_e2m1_to_float32(fp4):
    # TODO: implement following the FP8 E4M3 example above
    pass


# ── Sanity check — run after completing the functions ─────────────────
print('FP4 E2M1 sanity check:')
print(f"{'Original':>10} {'FP4 bits':>10} {'Reconstructed':>15} {'Error':>10}")
print('-' * 50)
for v in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 50.0, -1.5, -6.0]:
    try:
        fp4  = float32_to_fp4_e2m1(v)
        back = fp4_e2m1_to_float32(fp4)
        err  = abs(v - back)
        print(f'{v:>10.4f} {fp4:>10b}     {back:>10.4f}     {err:>10.6f}')
    except TypeError:
        print(f'{v:>10.4f}   ← implement the functions above first')